In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pickle
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, cross_val_predict, cross_val_score, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
import shap
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [2]:
import utils
ft_features, ft_properties = utils.processed_data('features.xlsx', 'properties.xlsx') 

In [5]:
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 13,
    "figure.dpi": 300,
    "savefig.dpi": 300
})

model_name = 'gradient_boost'
gbreg = GradientBoostingRegressor
with open(f'models/{model_name}_best_parameters.pkl','rb') as f:
    best_parameters = pickle.load(f)
print(best_parameters)
for col, parameters in best_parameters.items():
    print(col,'\n')

    y = ft_properties[col]
    X = ft_features

    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=22)
    X_train_df = pd.DataFrame(X_train,columns=ft_features.columns)
    
    model = gbreg(**parameters)
    model.fit(X_train, y_train)

    explainer = shap.Explainer(model, X_train_df)

    shap_values = explainer(X_train_df, check_additivity=False)
    plt.figure()
    shap.plots.beeswarm(shap_values, show=False)
    plt.xlabel('SHAP value (impact on model output)')
    plt.title(f"SHAP Beeswarm ({col})", pad=15)
    plt.tight_layout()
    plt.savefig(f'plots/SHAP_beeswarm_{model_name}_{col}.jpg', bbox_inches='tight')
    plt.close()

    # Bar plot
    plt.figure()
    shap.summary_plot(shap_values, X_train_df, plot_type="bar", show=False)
    plt.xlabel('Mean absolute SHAP value')
    plt.title(f"Feature Importance ({col})", pad=15)
    plt.tight_layout()
    plt.savefig(f'plots/SHAP_bar_{model_name}_{col}.jpg', bbox_inches='tight')
    plt.close()

{'CO(conv)': {'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 200, 'subsample': 0.6}, 'CH4': {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.9}, 'CO2': {'learning_rate': 0.01, 'max_depth': 2, 'n_estimators': 100, 'subsample': 0.7}, 'C2-C4': {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 300, 'subsample': 0.6}, 'С5+': {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 300, 'subsample': 0.6}, 'Mult': {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.6}}
CO(conv) 

CH4 

CO2 

C2-C4 

С5+ 

Mult 



In [3]:
def make_shap_plots(features, properties, model_instance, model_name, bar=True):
    
    plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 13,
    "figure.dpi": 300,
    "savefig.dpi": 300})
    
    with open(f'models/{model_name}_best_parameters.pkl','rb') as f:
        best_parameters = pickle.load(f)
    print(best_parameters)
    for col, parameters in best_parameters.items():
        print(col,'\n')

        y = properties[col]
        X = features

        from sklearn.model_selection import train_test_split
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=22)
        X_train_df = pd.DataFrame(X_train,columns=ft_features.columns)

        model = model_instance(**parameters)
        model.fit(X_train, y_train)

        explainer = shap.Explainer(model, X_train_df)

        shap_values = explainer(X_train_df, check_additivity=False)
        plt.figure()
        shap.plots.beeswarm(shap_values, show=False)
        plt.xlabel('SHAP value (impact on model output)')
        plt.title(f"SHAP Beeswarm ({col})", pad=15)
        plt.tight_layout()
        plt.savefig(f'plots/SHAP_beeswarm_{model_name}_{col}.jpg', bbox_inches='tight')
        plt.close()

        # Bar plot
        if bar==True:
            plt.figure()
            shap.summary_plot(shap_values, X_train_df, plot_type="bar", show=False)
            plt.xlabel('Mean absolute SHAP value')
            plt.title(f"Feature Importance ({col})", pad=15)
            plt.tight_layout()
            plt.savefig(f'plots/SHAP_bar_{model_name}_{col}.jpg', bbox_inches='tight')
            plt.close()

In [5]:
model_name = 'gradient_boost'
gbreg = GradientBoostingRegressor
make_shap_plots(ft_features, ft_properties, gbreg, model_name)

{'CO(conv)': {'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 200, 'subsample': 0.6}, 'CH4': {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.9}, 'CO2': {'learning_rate': 0.01, 'max_depth': 2, 'n_estimators': 100, 'subsample': 0.7}, 'C2-C4': {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 300, 'subsample': 0.6}, 'С5+': {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 300, 'subsample': 0.6}, 'Mult': {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.6}}
CO(conv) 

CH4 

CO2 

C2-C4 

С5+ 

Mult 

